# ObfusQA failure analysis: Colab runner

Run the cells **in order**. Each step prints a check; don't continue until it passes (see the Coding Guide).
All outputs go to Google Drive (`output_dir` in `config.json`), so a disconnect loses at most one batch.
Before running anything: commit `PREREGISTRATION.md` with its date.

## 0. GPU, Drive, code, packages

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Option A: your GitHub repo (recommended)
# !git clone https://github.com/Soumya080/obfusqa-failure-analysis.git
# Option B: upload obfusqa-failure-analysis.zip to Colab's file panel, then:
# !unzip -q obfusqa-failure-analysis.zip
%cd /content/obfusqa-failure-analysis
!pip -q install -r requirements.txt
!python -m spacy download en_core_web_sm -q

In [ ]:
# Needs a Hugging Face token with read access, and Llama 3.1 access approved on its model page.
from huggingface_hub import login
login()

## 1. Config and CPU tests (must all pass)

In [ ]:
import sys, json
sys.path.insert(0, '.')
import pandas as pd
from src import data, score, generate, padding, probes, stats, analysis, attribute, annotate, figures
from src.utils import load_config, results_dir, save_config_snapshot, write_df, read_df, read_jsonl

cfg = load_config()
OUT = results_dir(cfg)
print('outputs ->', OUT)
!python -m pytest -q tests/

## 2. Data: inspect, map columns, build the long table

In [ ]:
raw = data.load_raw(cfg['dataset_id'])
data.inspect(raw)
layout, guess = data.guess_columns(raw)
print('\nGUESS ->', layout, guess)

**Stop and check.** Does the guess map the right columns to base / NEI / DI / CO / answer?
If not, set them by hand in the next cell (e.g. `cfg['layout']='WIDE'; cfg['columns']['nei']='NEI'`).

In [ ]:
# cfg['layout'] = 'WIDE'
# cfg['columns'].update({'base': '...', 'nei': '...', 'di': '...', 'co': '...', 'answer': '...', 'aliases': None, 'id': None})
long = data.build_long_table(raw, cfg)
# If 'share of items with aliases' printed 0%, recover TriviaQA aliases (recommended):
# long = data.attach_triviaqa_aliases(long)
write_df(long, OUT / 'long.jsonl')
save_config_snapshot(cfg, OUT)
long.head(8)

In [ ]:
# Answer leakage: how often does the variant text literally contain the gold answer?
leak = analysis.answer_in_text(long)
leak['rate'] = leak['k'] / leak['n']
leak

## 3. Model and smoke test

In [ ]:
try:
    model, tok = generate.load_model(cfg)
    MODEL_NAME = cfg['model_id']
except Exception as e:
    print('Primary model failed (gated? no access yet?):', e)
    model, tok = generate.load_model(cfg, cfg['fallback_model_id'])
    MODEL_NAME = cfg['fallback_model_id']

In [ ]:
mk = generate.make_job
base20 = long[long.condition == 'BASE'].head(20)
smoke = [mk(r.item_id, 'SMOKE', cfg['answer_prompt'].format(question=r.text), tok) for r in base20.itertuples()]
rows = generate.run_jobs(smoke, model, tok, OUT / 'smoke.jsonl', batch_size=cfg['batch_size'], max_new_tokens=cfg['max_new_tokens'])
for r, b in zip(rows, base20.itertuples()):
    print(f"{b.gold!r:30}  <-  {score.clean_prediction(r['raw_output'])!r}")
generate.determinism_check(smoke[:10], model, tok)

**Check:** answers are short (not sentences), most look right, determinism says 10 / 10.

## 4. Main run: BASE, NEI, DI, CO (C1-C4)

In [ ]:
main_jobs = [mk(r.item_id, r.condition, cfg['answer_prompt'].format(question=r.text), tok) for r in long.itertuples()]
main_rows = generate.run_jobs(main_jobs, model, tok, OUT / 'main.jsonl', batch_size=cfg['batch_size'], max_new_tokens=cfg['max_new_tokens'])
scored = analysis.score_outputs(main_rows, long)
scored.groupby('condition')[['em', 'contains']].mean().reindex(['BASE', 'NEI', 'DI', 'CO'])

**Check:** roughly BASE > NEI > DI / CO. If BASE is very low, look at 20 raw outputs: is it the prompt or the scorer?

## 5. C5: BASE padded to CO length

In [ ]:
pool = padding.load_sentence_pool(cfg)
c5, pad_report = padding.build_padded(long, tok, pool, tolerance=cfg['padding']['tolerance'], seed=cfg['seed'])
assert padding.leak_check(c5) == 0
write_df(c5, OUT / 'c5.jsonl'); pad_report.to_csv(OUT / 'padding_report.csv', index=False)
print(c5.text.iloc[0][:600])

In [ ]:
c5_jobs = [mk(r.item_id, 'C5', cfg['answer_prompt'].format(question=r.text), tok) for r in c5.itertuples()]
c5_rows = generate.run_jobs(c5_jobs, model, tok, OUT / 'c5_out.jsonl', batch_size=cfg['batch_size'], max_new_tokens=cfg['max_new_tokens'])

## 6. Gold entities for NEI (manual check required)

In [ ]:
ents = probes.guess_gold_entities(long)
ents.to_csv(OUT / 'gold_entities_to_check.csv', index=False)
ents.head(10)

**Manual step.** Open `gold_entities_to_check.csv` (in your Drive folder) in Google Sheets.
For every row, check `gold_entity` is the thing the NEI question describes; fix it if not; set `checked` = 1.
Download as CSV and save it to the same folder as `gold_entities_checked.csv`. Check at least the first 60 rows
carefully; skim the rest.

In [ ]:
ents_checked = pd.read_csv(OUT / 'gold_entities_checked.csv').fillna('')
print('checked rows:', int(ents_checked['checked'].sum()), '/', len(ents_checked))

## 7. Probes: P1 (resolve), I1 (own hint), I2 (oracle hint)

In [ ]:
p1_jobs = probes.resolution_jobs(long, cfg, tok, mk)
p1_rows = generate.run_jobs(p1_jobs, model, tok, OUT / 'p1.jsonl', batch_size=cfg['batch_size'], max_new_tokens=cfg['max_new_tokens'])
p1 = probes.score_resolution(p1_rows, ents_checked)
print('P1 resolution accuracy:', round(p1.p1_correct.mean(), 3))
own = dict(zip(p1.item_id, p1.p1_pred))
gold_ent = dict(zip(ents_checked.item_id, ents_checked.gold_entity))
hint_jobs = probes.hint_jobs(long, own, cfg, tok, mk, 'I1') + probes.hint_jobs(long, gold_ent, cfg, tok, mk, 'I2')
hint_rows = generate.run_jobs(hint_jobs, model, tok, OUT / 'hints.jsonl', batch_size=cfg['batch_size'], max_new_tokens=cfg['max_new_tokens'])

## 8. Results tables

In [ ]:
all_rows = read_jsonl(OUT / 'main.jsonl') + read_jsonl(OUT / 'c5_out.jsonl') + read_jsonl(OUT / 'hints.jsonl')
scored = analysis.score_outputs(all_rows, long)
w = analysis.wide_table(scored)
nb = cfg['bootstrap_resamples']
t1 = analysis.main_table(w, n_boot=nb);            t1.to_csv(OUT / 'table1_accuracy.csv', index=False)
ls = analysis.length_result(w, n_boot=nb);         json.dump(ls, open(OUT / 'length_share.json', 'w'), indent=2)
t2 = analysis.composition_table(w, p1, n_boot=nb); t2.to_csv(OUT / 'table2_composition.csv', index=False)
print('Accuracy:', w[[c for c in w.columns if c.startswith('em_')]].mean().round(3).to_dict())
print('Length share of CO drop:', ls)
display(t1.round(3)); display(t2.round(3))

## 9. (Optional) selection probe: 5 samples on failed items

In [ ]:
SAMPLED = None
if cfg['sampling']['enabled']:
    fails = []
    for cond in ['NEI', 'DI', 'CO']:
        ids = w[(w.em_BASE == 1) & (w[f'em_{cond}'] == 0)].index
        txt = long.set_index(['item_id', 'condition'])['text']
        fails += [mk(i, f'S_{cond}', cfg['answer_prompt'].format(question=txt[(i, cond)]), tok) for i in ids]
    s_rows = generate.run_jobs(fails, model, tok, OUT / 'samples.jsonl', batch_size=4, max_new_tokens=cfg['max_new_tokens'],
                               do_sample=True, temperature=cfg['sampling']['temperature'],
                               num_return_sequences=cfg['sampling']['n_samples'])
    g = long.drop_duplicates('item_id').set_index('item_id')
    SAMPLED = {(r['item_id'], r['condition'][2:]): any(score.exact_match(o, g.loc[r['item_id'], 'gold'], g.loc[r['item_id'], 'aliases'])
               for o in r['raw_outputs']) for r in s_rows}

## 10. Attribution and Figure 1

In [ ]:
att = attribute.attribute(w, long, p1=p1, sampled_hits=SAMPLED)
summ = attribute.attribution_summary(att)
att.to_csv(OUT / 'attribution_items.csv', index=False); summ.to_csv(OUT / 'attribution_summary.csv', index=False)
figures.attribution_figure(summ, OUT / 'figure1_attribution.png', model_name=MODEL_NAME.split('/')[-1])
summ.round(3)

## 11. Export annotation sheets (RQ4 and CO audit)

In [ ]:
annotate.export_failure_sample(att, long, OUT / 'failures_to_label.csv', n=60, seed=cfg['seed'])
annotate.export_co_audit(long, OUT / 'co_audit.csv', n=50, seed=cfg['seed'])

**Manual step.** Label both sheets with a classmate, independently, following `ANNOTATION_GUIDE.md`.
Hide the `auto_cause_hidden` column while labelling. Save the filled files back to the Drive folder
as `failures_labelled.csv` and `co_audit_labelled.csv`.

In [ ]:
k1 = annotate.kappa_report(OUT / 'failures_labelled.csv')
k2 = annotate.kappa_report(OUT / 'co_audit_labelled.csv', a='false_claim_A', b='false_claim_B', final='__none__', auto='__none__')
co = pd.read_csv(OUT / 'co_audit_labelled.csv')
final = co[['false_claim_A', 'false_claim_B']].max(axis=1)   # or adjudicate disagreements by hand
print('CO items with a false/misleading claim:', stats.wilson_ci(int(final.sum()), int(final.notna().sum())))